# Sales Analytics Aggregation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round, sum as spark_sum

spark = SparkSession.builder.appName("AnalyticsView").getOrCreate()

BUCKET_NAME = "ecommerce-data-pipeline-jayant"
DIM_PATH = f"s3://{BUCKET_NAME}/warehouse/dimensions/"
FACT_PATH = f"s3://{BUCKET_NAME}/warehouse/facts/"
ANALYTICS_PATH = f"s3://{BUCKET_NAME}/warehouse/analytics/sales_summary/"

dim_customer = spark.read.parquet(DIM_PATH + "dim_customer/")
dim_product = spark.read.parquet(DIM_PATH + "dim_product/")
dim_region = spark.read.parquet(DIM_PATH + "dim_region/")
dim_date = spark.read.parquet(DIM_PATH + "dim_date/")
fact_sales = spark.read.parquet(FACT_PATH + "fact_sales/")

### Join Fact and Dimension Tables

In [ ]:
analytics_df = (
    fact_sales
    .join(dim_customer, on="customer_id", how="left")
    .join(dim_product, on="product_id", how="left")
    .join(dim_region, on="region_id", how="left")
    .join(dim_date, on="order_date", how="left")
)

### Aggregate Monthly Sales KPIs

In [ ]:
sales_summary = (
    analytics_df
    .groupBy("region", "category", "year", "month")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity")
    )
)

sales_summary.write.mode("overwrite").parquet(ANALYTICS_PATH)